# Assignment 3: Tool-Enhanced Pricing Agent with Optimizer

## Objective
Add **external tool capabilities** to our pricing agent, allowing it to call specialized pricing optimization functions for more accurate and deterministic calculations.

## Requirements
**Tools Added:**
- Margin Calculator Tool
- Elasticity Adjustment Tool
- Competitor Matching Tool

**Agent Flow:**
1. Agent interprets inputs
2. Calls appropriate tools
3. Computes recommended price
4. Returns final explanation + number

**Tool Example:**
```python
optimizer.calculate_price(
    cost_price=400,
    target_margin=0.25,
    competitor_price=579,
    elasticity="medium"
)
```

## Setup & Dependencies

Install required packages for tool-enhanced agent implementation.

In [1]:
# Install required packages for tool-enhanced agent
!pip install -q langchain langchain-groq langchain-community
!pip install -q pandas numpy scipy
!pip install -q pydantic

In [2]:
# Import required libraries
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
import os
import getpass
import pandas as pd
import json
import math
from typing import List, Dict, Optional, Union
from pydantic import BaseModel

In [3]:
# Set up your Groq API key
print("Please enter your Groq API key:")
print("(You can get one free at: https://console.groq.com/)")
groq_api_key = getpass.getpass("Groq API Key: ")
os.environ["GROQ_API_KEY"] = groq_api_key
print("API key set successfully!")

Please enter your Groq API key:
(You can get one free at: https://console.groq.com/)
API key set successfully!


## Pricing Optimization Tools

Create the three external tools our agent will use for pricing calculations.

In [4]:
@tool
def margin_calculator(cost_price: float, target_margin: float) -> dict:
    """
    Calculate optimal selling price based on cost and target margin.
    
    Args:
        cost_price (float): The cost to produce/acquire the product
        target_margin (float): Desired margin as decimal (e.g., 0.25 for 25%)
    
    Returns:
        dict: Contains selling_price, margin_dollar, and validation info
    """
    price = cost_price / (1 - target_margin)
    margin_dollar = price - cost_price
    validation = {
        "valid": True,
        "message": ""
    }
    if target_margin >= 1.0:
        validation["valid"] = False
        validation["message"] = "Target margin must be less than 1.0"
    elif target_margin < 0:
        validation["valid"] = False
        validation["message"] = "Target margin must be non-negative"

    return {
        "selling_price": price,
        "margin_dollar": margin_dollar,
        "validation": validation
    }

@tool
def elasticity_adjustment(base_price: float, elasticity: str, price_change_percent: float) -> dict:
    """
    Adjust pricing based on demand elasticity to optimize revenue.
    
    Args:
        base_price (float): Starting price point
        elasticity (str): Elasticity level - 'low', 'medium', 'medium-high', or 'high'
        price_change_percent (float): Proposed price change as percentage (e.g., 10 for 10% increase)
    
    Returns:
        dict: Contains adjusted_price, demand_impact, revenue_impact
    """
    elasticity_coefficients = {
        'low': -0.5,
        'medium': -1.0,
        'medium-high': -1.5,
        'high': -2.0
    }
    demand_change = elasticity_coefficients.get(elasticity, 0) * (price_change_percent / 100)
    adjusted_price = base_price * (1 + price_change_percent / 100)
    revenue_impact = (1 + price_change_percent / 100) * (1 + demand_change) - 1
    
    return {
        "adjusted_price": adjusted_price,
        "demand_impact": demand_change,
        "revenue_impact": revenue_impact
    }
    


@tool
def competitor_matching(our_cost: float, competitor_price: float, positioning: str = "match") -> dict:
    """
    Calculate optimal pricing relative to competitor prices.
    
    Args:
        our_cost (float): Our cost to produce the product
        competitor_price (float): Competitor's selling price
        positioning (str): Strategy - 'undercut', 'match', 'premium', or 'aggressive'
    
    Returns:
        dict: Contains recommended_price, our_margin, competitive_analysis
    """
    
    positioning_strategies = {
        'aggressive': 0.85,
        'undercut': 0.92,
        'match': 0.98,
        'premium': 1.10
    }

    multiplier = positioning_strategies.get(positioning, 0.98)
    recommended_price = competitor_price * multiplier
    our_margin = recommended_price - our_cost
    competitive_analysis = {
        "positioning": positioning,
        "multiplier": multiplier,
        "recommended_price": recommended_price,
        "our_margin": our_margin,
        "covers_cost": recommended_price > our_cost
    }   
    
    return {
        "recommended_price": recommended_price,
        "our_margin": our_margin,
        "competitive_analysis": competitive_analysis
    }

print("Testing Pricing Tools:")
print("="*40)

margin_result = margin_calculator.invoke({"cost_price": 400, "target_margin": 0.25})
print(f"Margin test: {margin_result}")

elasticity_result = elasticity_adjustment.invoke({"base_price": 533, "elasticity": "medium", "price_change_percent": 8.5})
print(f"Elasticity test: {elasticity_result}")

competitor_result = competitor_matching.invoke({"our_cost": 400, "competitor_price": 579, "positioning": "match"})
print(f"Competitor test: {competitor_result}")

print("Tools implementation pending...")

Testing Pricing Tools:
Margin test: {'selling_price': 533.3333333333334, 'margin_dollar': 133.33333333333337, 'validation': {'valid': True, 'message': ''}}
Elasticity test: {'adjusted_price': 578.305, 'demand_impact': -0.085, 'revenue_impact': -0.007225000000000037}
Competitor test: {'recommended_price': 567.42, 'our_margin': 167.41999999999996, 'competitive_analysis': {'positioning': 'match', 'multiplier': 0.98, 'recommended_price': 567.42, 'our_margin': 167.41999999999996, 'covers_cost': True}}
Tools implementation pending...


## Tool-Enhanced Pricing Agent

Create our enhanced pricing agent that can call these external tools.

In [10]:
class ToolEnhancedPricingAgent:
    def __init__(self):
        self.llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.1, max_tokens=2048)
        
        self.tools = [margin_calculator, elasticity_adjustment, competitor_matching]
        
        self.llm_with_tools = self.llm.bind_tools(self.tools)
        
        # TODO: Create system message for tool-enhanced agent
        self.system_message = SystemMessage(content="""
        You are a pricing optimization agent that uses tools to calculate optimal prices based on cost, competitor pricing, and demand elasticity.  
        1. Available tools: margin_calculator, elasticity_adjustment, competitor_matching
        2. Process: Analyze request → Call tools → Interpret results → Recommend price
        3. Always use tools for calculations rather than manual estimates
        """)
        
    def calculate_price(self, cost_price: float, target_margin: float = None, 
                       competitor_price: float = None, elasticity: str = None,
                       positioning: str = "match") -> str:
        """
        Calculate optimal price using available tools
        """
        
        request_parts = [
            f"Calculate optimal pricing for a product with cost ${cost_price}"
        ]
        
        if target_margin: request_parts.append(f"Target margin: {target_margin*100:.1f}%")
        if competitor_price: request_parts.append(f"Competitor price: ${competitor_price}")
        if elasticity: request_parts.append(f"Demand elasticity: {elasticity}")
        if positioning: request_parts.append(f"Positioning strategy: {positioning}")
        
        request_parts.append("\nUse your tools to calculate the optimal price and provide detailed analysis.")
        
        human_message = HumanMessage(content="\n".join(request_parts))
        messages = [self.system_message, human_message]
        response = self.llm_with_tools.invoke(messages)
        messages.append(response)
        
        while response.tool_calls:
            for tool_call in response.tool_calls:
                tool_fn = next((t for t in self.tools if t.name == tool_call["name"]), None)
                if tool_fn:
                    tool_result = tool_fn.invoke(tool_call["args"])
                    messages.append(ToolMessage(
                        content=str(tool_result),
                        tool_call_id=tool_call["id"]
                    ))
            response = self.llm_with_tools.invoke(messages)
            messages.append(response)
        
        return response.content

# Initialize the tool-enhanced agent
print("Initializing Tool-Enhanced Pricing Agent...")
optimizer = ToolEnhancedPricingAgent()
print("Agent implementation pending...")
print(f"Available tools: {[tool.name for tool in optimizer.tools]}")

Initializing Tool-Enhanced Pricing Agent...
Agent implementation pending...
Available tools: ['margin_calculator', 'elasticity_adjustment', 'competitor_matching']


## Testing the Tool-Enhanced Agent

Test our agent with the assignment example and various scenarios.

In [12]:
# Test with the assignment example
print("Assignment Example: Tool-Enhanced Pricing")
print("="*50)

result = optimizer.calculate_price(
    cost_price=400,
    target_margin=0.25,
    competitor_price=579, 
    elasticity="medium"
)

print(result)

print("Testing pending - implement the agent first")

Assignment Example: Tool-Enhanced Pricing
Based on the analysis, the optimal price for the product is $533.33, which provides a 25% margin. This price is calculated using the margin calculator tool.

The competitor matching tool suggests a price of $567.42, which is a 98% of the competitor's price. This price provides a margin of 16.74% and covers the cost.

The elasticity adjustment tool suggests no price change, as the demand elasticity is medium and the price change is 0%.

Therefore, the recommended price for the product is $567.42, which provides a good balance between margin and competitiveness.
Testing pending - implement the agent first


In [16]:
# Create comprehensive analysis method
def get_comprehensive_analysis(product_name: str, category: str, 
                             cost_price: float, target_margin: float = None,
                             competitor_price: float = None, elasticity: str = None) -> str:
    """
    Get comprehensive pricing analysis using multiple tools
    """

    request_parts = [
        f"Calculate comprehensive pricing analyses with product name {product_name}, category {category} and cost price ${cost_price}"
    ]
    
    if target_margin is not None: request_parts.append(f"Target margin: {target_margin*100:.1f}%")
    if competitor_price is not None: request_parts.append(f"Competitor price: ${competitor_price}")
    if elasticity: request_parts.append(f"Demand elasticity: {elasticity}")
    
    request_parts.append("\nUse above tools to calculate comprehensive pricing analyses")
    
    human_message = HumanMessage(content="\n".join(request_parts))
    messages = [optimizer.system_message, human_message]
    response = optimizer.llm_with_tools.invoke(messages)
    messages.append(response)
        
    while response.tool_calls:
        for tool_call in response.tool_calls:
            tool_fn = next((t for t in optimizer.tools if t.name == tool_call["name"]), None)
            if tool_fn:
                tool_result = tool_fn.invoke(tool_call["args"])
                messages.append(ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                ))
        response = optimizer.llm_with_tools.invoke(messages)
        messages.append(response)
    
    return response.content  # outside the while loop

# Test comprehensive analysis
print("Comprehensive Analysis: Puma Sneakers")
print("="*45)

comprehensive_result = get_comprehensive_analysis(
    product_name="Puma Training Sneakers",
    category="Footwear",
    cost_price=400,
    target_margin=0.25,
    competitor_price=579,
    elasticity="medium"
)

print(comprehensive_result)


Comprehensive Analysis: Puma Sneakers
Based on the comprehensive pricing analyses, the recommended price for the Puma Training Sneakers is $567.42. This price is calculated by matching the competitor's price, which is $579. The margin calculator tool suggests a selling price of $533.33 with a target margin of 25%. However, considering the competitor's price, the competitor matching tool recommends a price of $567.42, which covers the cost and provides a margin of 41.7%.

The elasticity adjustment tool suggests adjusting the price by 10% to account for medium demand elasticity. However, this adjustment would result in a price of $636.90, which is higher than the competitor's price and may not be optimal.

Therefore, the recommended price for the Puma Training Sneakers is $567.42, which balances the need to cover costs, provide a margin, and remain competitive in the market.


## Testing Different Scenarios

Test various pricing scenarios to see how tools enhance decision-making.

In [ ]:
print("Scenario 1: High Elasticity Electronics")
print("="*40)

electronics_result = get_comprehensive_analysis(
    product_name="Gaming Laptop",
    category="Electronics", 
    cost_price=750,
    target_margin=0.18,
    competitor_price=950,
    elasticity="high"
)

print(electronics_result)

print("Scenario testing pending...")

Scenario 1: High Elasticity Electronics
Based on the comprehensive pricing analyses, the recommended price for the Gaming Laptop is $931. This price is calculated by matching the competitor's price of $950, which results in a margin of 18.1%. The elasticity adjustment tool suggests that the current price of $950 is optimal, with no price change needed to maximize revenue.

The margin calculator tool indicates that the optimal selling price is $914.63, which would result in a margin of $164.63. However, this price is lower than the competitor's price, so it may not be the best option for the company.

Overall, the recommended price of $931 appears to be a good balance between maximizing revenue and maintaining a competitive price.
Scenario testing pending...


In [ ]:
print("Scenario 2: Low Elasticity Luxury Goods")
print("="*42)

luxury_result = get_comprehensive_analysis(
    product_name="Designer Watch",
    category="Luxury Goods",
    cost_price=300,
    target_margin=0.60,
    competitor_price=850,
    elasticity="low"
)

print(luxury_result)

print("Scenario testing pending...")

Scenario 2: Low Elasticity Luxury Goods
Based on the comprehensive pricing analyses, the recommended price for the Designer Watch is $833. This price is calculated by matching the competitor's price of $850, which results in a margin of 53.3%. The margin calculator tool suggests a selling price of $750 with a margin of $450, but this is not taken into account in the competitor matching analysis. The elasticity adjustment tool suggests no change in price due to low demand elasticity. Therefore, the final recommended price is $833.
Scenario testing pending...


In [ ]:
print("Scenario 3: Aggressive Market Entry")
print("="*35)

aggressive_result = optimizer.calculate_price(
    cost_price=60,
    competitor_price=95,
    positioning="aggressive",
    elasticity="medium-high"
)

print(aggressive_result)

print("Aggressive pricing testing pending...")

Scenario 3: Aggressive Market Entry
Based on the analysis, the optimal price for the product is $80.75. This price is calculated by applying an aggressive positioning strategy relative to the competitor's price of $95. The competitor matching tool recommends a price that is 85% of the competitor's price, which results in a price of $80.75.

The elasticity adjustment tool is used to analyze the demand elasticity of the product. The tool determines that the demand elasticity is medium-high, which means that small price changes will have a significant impact on demand. However, in this case, the price change is 0%, so the demand impact is 0.

The margin calculator tool is used to calculate the optimal selling price based on the cost and target margin. The tool determines that the optimal selling price is $60.00, which is the same as the cost price. This is because the target margin is 0%.

Overall, the optimal price for the product is $80.75, which is a balance between the competitor's pr

## Tool Performance Analysis

Analyze how tools improve pricing accuracy and decision-making.

In [ ]:
def analyze_tool_effectiveness():
    """Compare tool-enhanced vs manual calculation approaches"""
    
    print("Tool Effectiveness Analysis")
    print("="*30)
    
    test_cases = [
        {
            "name": "Standard Margin Calculation",
            "cost": 100,
            "target_margin": 0.30,
            "manual_estimate": 130,  # Common error: adding 30% instead of dividing
            "tool_function": "margin_calculator"
        },
        {
            "name": "Elasticity Adjustment",
            "base_price": 200,
            "elasticity": "medium",
            "price_change_percent": 10,
            "manual_estimate": 220,  # Common error: not accounting for demand change
            "tool_function": "elasticity_adjustment"
        },
        {
            "name": "Competitor Matching - Undercut",
            "our_cost": 50,
            "competitor_price": 80,
            "positioning": "undercut",
            "manual_estimate": 78,  # Common error: undercutting by fixed amount instead of percentage
            "tool_function": "competitor_matching"
        }
    ]

    def call_appropriate_tool(case: dict):
        if case["tool_function"] == "margin_calculator":
            return margin_calculator.invoke({
            "cost_price": case["cost"],
            "target_margin": case["target_margin"]
        })
        elif case["tool_function"] == "elasticity_adjustment":
            return elasticity_adjustment.invoke({
            "base_price": case["base_price"],
            "elasticity": case["elasticity"],
            "price_change_percent": case["price_change_percent"]
        })
        elif case["tool_function"] == "competitor_matching":
            return competitor_matching.invoke({
            "our_cost": case["our_cost"],
            "competitor_price": case["competitor_price"],
            "positioning": case["positioning"]
        }) 
    
    for i, case in enumerate(test_cases, 1):
        print(f"\nTest Case {i}: {case['name']}")
        print("-" * 30)
        
        print(f"Manual estimate: ${case['manual_estimate']}")
        tool_result = call_appropriate_tool(case)
        print(f"Tool result: {tool_result}")
        print("Tool comparison pending...")

   

# Run analysis
analyze_tool_effectiveness()

Tool Effectiveness Analysis

Test Case 1: Standard Margin Calculation
------------------------------
Manual estimate: $130
Tool result: {'selling_price': 142.85714285714286, 'margin_dollar': 42.85714285714286, 'validation': {'valid': True, 'message': ''}}
Tool comparison pending...

Test Case 2: Elasticity Adjustment
------------------------------
Manual estimate: $220
Tool result: {'adjusted_price': 220.00000000000003, 'demand_impact': -0.1, 'revenue_impact': -0.009999999999999898}
Tool comparison pending...

Test Case 3: Competitor Matching - Undercut
------------------------------
Manual estimate: $78
Tool result: {'recommended_price': 73.60000000000001, 'our_margin': 23.60000000000001, 'competitive_analysis': {'positioning': 'undercut', 'multiplier': 0.92, 'recommended_price': 73.60000000000001, 'our_margin': 23.60000000000001, 'covers_cost': True}}
Tool comparison pending...


## Interactive Pricing Optimizer

Create an interactive tool to test different pricing scenarios.

In [ ]:
def interactive_pricing_optimizer():
    """
    Interactive pricing consultation with tool-enhanced agent
    """
    
    print("Interactive Tool-Enhanced Pricing Optimizer")
    print("="*45)
    print("Available elasticity levels: low, medium, medium-high, high")
    print("Positioning strategies: aggressive, undercut, match, premium")
    print()
    
    # TODO: Collect inputs from user
    # product = input("Product name: ")
    # category = input("Category: ")
    # cost = float(input("Cost price ($): "))
    # etc.
    
    print("Interactive mode commented out - uncomment to use")
    
    # TODO: Get comprehensive analysis with collected inputs
    # analysis = get_comprehensive_analysis(...)
    # print(analysis)

# Uncomment to run interactive optimizer
# interactive_pricing_optimizer()

## Tool Comparison Study

Compare different pricing approaches using our tools.

In [ ]:
def pricing_strategy_comparison(cost: float, target_margin: float, competitor: float, elasticity: str):
    """
    Compare different pricing strategies using tools
    """
    
    print(f"Pricing Strategy Comparison")
    print(f"Cost: ${cost}, Target Margin: {target_margin*100:.0f}%, Competitor: ${competitor}, Elasticity: {elasticity}")
    print("="*80)
    
    # TODO: Compare different strategies
    # Strategy 1: Margin-based pricing
    # margin_result = margin_calculator.invoke(...)
    
    # Strategy 2: Competitive pricing strategies
    # for strategy in ['aggressive', 'undercut', 'match', 'premium']:
    #     comp_result = competitor_matching.invoke(...)
    
    # Strategy 3: Elasticity-optimized pricing  
    # Test different price adjustments with elasticity_adjustment
    
    print("Strategy comparison implementation pending...")

# Test pricing strategy comparison
print("Example: Footwear Pricing Strategy Comparison")
pricing_strategy_comparison(
    cost=90,
    target_margin=0.35,
    competitor=160,
    elasticity="medium-high"
)

## Edge Case Testing

Test how tools handle challenging pricing scenarios.

In [ ]:
def test_edge_cases():
    """
    Test edge cases and error handling in pricing tools
    """
    
    print("Edge Case Testing")
    print("="*20)
    
    # TODO: Test edge cases
    # Edge Case 1: Impossible margin (90%)
    # edge1 = margin_calculator.invoke({"cost_price": 100, "target_margin": 0.90})
    
    # Edge Case 2: Invalid elasticity
    # edge2 = elasticity_adjustment.invoke({"base_price": 100, "elasticity": "super-high", "price_change_percent": 10})
    
    # Edge Case 3: Competitor price below cost
    # edge3 = competitor_matching.invoke({"our_cost": 200, "competitor_price": 150, "positioning": "match"})
    
    # Edge Case 4: Extreme price elasticity impact
    # edge4 = elasticity_adjustment.invoke({"base_price": 100, "elasticity": "high", "price_change_percent": 50})
    
    print("Edge case testing implementation pending...")

# Run edge case tests
test_edge_cases()

## Your Turn: Implement the Tools and Agent

**Step 1:** Implement the three pricing tools:
- `margin_calculator`: Calculate price from cost and margin
- `elasticity_adjustment`: Calculate demand/revenue impact of price changes
- `competitor_matching`: Calculate competitive pricing strategies

**Step 2:** Implement the `ToolEnhancedPricingAgent`:
- Initialize LLM with tool binding
- Create tool calling logic
- Process tool results and generate recommendations

**Step 3:** Test with the assignment example:
```python
optimizer.calculate_price(
    cost_price=400,
    target_margin=0.25,
    competitor_price=579,
    elasticity="medium"
)
```

**Step 4:** Extend with additional tools and scenarios

In [ ]:
# Your implementation space

# TODO: Implement margin_calculator tool
# TODO: Implement elasticity_adjustment tool
# TODO: Implement competitor_matching tool
# TODO: Implement ToolEnhancedPricingAgent class
# TODO: Test with assignment example
# TODO: Add additional test cases

print("Ready for your implementation!")
print("Follow the TODOs above to build the tool-enhanced pricing agent.")